# 06. 멀티모달 검색 (Multimodal Search)

PDF 데이터셋을 Blob Storage에 업로드하고, 텍스트와 이미지를 함께 인덱싱하여 멀티모달 검색을 구현합니다.

## 전체 흐름

```
[로컬 PDF 파일]
    │ 2. Blob 업로드
    ▼
[Blob Storage — pdf-documents 컨테이너]
    │ 3. 처리
    ▼
[Document Intelligence Layout]  →  텍스트 청크
    │
    ├─ 텍스트 → text-embedding-3-large → AI Search Index
    │
    └─ 이미지(Figure) → GPT-5.4 Verbalization → text-embedding-3-large → AI Search Index
                            │
                        [processed-documents] ← 추출 이미지(.png) 저장
```

## 참고
- [Multimodal search overview](https://learn.microsoft.com/en-us/azure/search/multimodal-search-overview)

## 1. 환경 설정

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

required_vars = [
    'AZURE_SEARCH_SERVICE_ENDPOINT',
    'AZURE_OPENAI_ENDPOINT',
    'AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT',
    'AZURE_STORAGE_ACCOUNT_NAME',
]
for var in required_vars:
    assert os.environ.get(var), f'{var} 환경변수가 설정되지 않았습니다.'

STORAGE_NAME     = os.environ['AZURE_STORAGE_ACCOUNT_NAME']
PDF_CONTAINER    = 'pdf-documents'      # 사용자가 업로드하는 PDF 컨테이너
ARTIFACT_CONTAINER = 'processed-documents'  # 추출 이미지 저장
MULTIMODAL_INDEX_NAME = 'multimodal-documents-index'

print('환경 설정 완료')
print(f"  AI Search : {os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']}")
print(f"  OpenAI    : {os.environ['AZURE_OPENAI_ENDPOINT']}")
print(f"  Doc Intel : {os.environ['AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT']}")
print(f"  Storage   : {STORAGE_NAME}")
print(f"  PDF 컨테이너: {PDF_CONTAINER}")

## 2. PDF 데이터셋 업로드

로컬 PDF 파일을 Blob Storage(`pdf-documents` 컨테이너)에 업로드합니다.

> **업로드 경로 설정**: 아래 `PDF_SOURCE`에 로컬 PDF 파일이 있는 디렉토리 또는 개별 파일 경로를 지정하세요.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

credential = DefaultAzureCredential()
blob_service = BlobServiceClient(
    account_url=f"https://{STORAGE_NAME}.blob.core.windows.net",
    credential=credential,
)

# ── 컨테이너 준비 ──────────────────────────────────────────────
for cname in [PDF_CONTAINER, ARTIFACT_CONTAINER]:
    try:
        blob_service.create_container(cname)
        print(f"컨테이너 생성: {cname}")
    except Exception:
        print(f"컨테이너 기사용: {cname}")

pdf_container = blob_service.get_container_client(PDF_CONTAINER)

In [ ]:
# ── 업로드할 PDF 경로 지정 ─────────────────────────────────────
# 디렉토리를 지정하면 하위 .pdf 파일 전체를 업로드합니다.
# 개별 파일 경로를 리스트로 지정할 수도 있습니다.

PDF_SOURCE = '../data/pdf'   # ← 여기에 로컬 PDF 경로 입력

source = Path(PDF_SOURCE)
if source.is_dir():
    local_pdfs = sorted(source.rglob('*.pdf'))
elif source.is_file() and source.suffix == '.pdf':
    local_pdfs = [source]
else:
    local_pdfs = []

print(f"업로드 대상: {len(local_pdfs)}개")
for p in local_pdfs:
    print(f"  {p}")

In [ ]:
# ── Blob Storage에 업로드 ─────────────────────────────────────
uploaded = []
for pdf_path in local_pdfs:
    blob_name = pdf_path.name
    with open(pdf_path, 'rb') as f:
        pdf_container.upload_blob(name=blob_name, data=f, overwrite=True)
    size_kb = pdf_path.stat().st_size // 1024
    print(f"  ✓ {blob_name} ({size_kb:,} KB)")
    uploaded.append(blob_name)

print(f"\n총 {len(uploaded)}개 파일 업로드 완료 → {PDF_CONTAINER}")

In [ ]:
# ── 컨테이너 내 PDF 목록 확인 ────────────────────────────────
pdf_blobs = [b for b in pdf_container.list_blobs() if b.name.endswith('.pdf')]
print(f"pdf-documents 컨테이너: {len(pdf_blobs)}개")
for b in pdf_blobs:
    print(f"  {b.name}  ({b.size:,} bytes)")

## 3. 멀티모달 인덱스 생성

텍스트와 이미지를 함께 저장하는 AI Search 인덱스를 생성합니다.
- `content_type` 필드로 text/image 구분
- `content_embedding`: text-embedding-3-large (3072 dims)
- `image_path`: 추출 이미지 Blob 경로

In [ ]:
from src.search.multimodal_index import MultimodalIndexManager

MULTIMODAL_INDEX_NAME = 'multimodal-documents-index'

mm_index = MultimodalIndexManager(
    endpoint=os.environ['AZURE_SEARCH_SERVICE_ENDPOINT'],
    admin_key=os.environ.get('AZURE_SEARCH_ADMIN_KEY'),
    index_name=MULTIMODAL_INDEX_NAME,
)

# 인덱스 생성 (text-embedding-3-large = 3072 dims)
mm_index.create_index(dimensions=3072)
print(f'멀티모달 인덱스 생성 완료: {MULTIMODAL_INDEX_NAME}')

## 4. Blob Storage PDF 처리 및 인덱싱

`pdf-documents` 컨테이너의 PDF를 순서대로 처리합니다.

### 처리 과정
1. Blob에서 PDF 다운로드
2. Document Intelligence Layout → 텍스트 청크 + Figure 이미지 추출
3. 텍스트 청크 → text-embedding-3-large 임베딩
4. Figure 이미지 → Blob 저장 + GPT-5.4 Verbalization → 임베딩
5. AI Search 인덱스에 업로드

In [ ]:
from src.preprocessing.multimodal_processor import MultimodalProcessor
from src.search.multimodal_index import MultimodalIndexManager

processor = MultimodalProcessor(
    di_endpoint=os.environ['AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT'],
    openai_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    storage_account_name=STORAGE_NAME,
    openai_api_key=os.environ.get('AZURE_OPENAI_API_KEY'),
    gpt_deployment=os.environ.get('AZURE_OPENAI_GPT54_DEPLOYMENT', 'gpt-5.4'),
    embedding_deployment=os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large'),
    artifacts_container=ARTIFACT_CONTAINER,
)

mm_index = MultimodalIndexManager(
    endpoint=os.environ['AZURE_SEARCH_SERVICE_ENDPOINT'],
    admin_key=os.environ.get('AZURE_SEARCH_ADMIN_KEY'),
    index_name=MULTIMODAL_INDEX_NAME,
)
mm_index.create_index(dimensions=3072)

print('프로세서 및 인덱스 준비 완료')

### 4-1. Blob에서 PDF 목록 조회

In [ ]:
# pdf-documents 컨테이너에서 PDF 목록 가져오기
pdf_blobs = [b for b in pdf_container.list_blobs() if b.name.endswith('.pdf')]

print(f"처리할 PDF: {len(pdf_blobs)}개")
for b in pdf_blobs:
    print(f"  {b.name}  ({b.size:,} bytes)")

### 4-2. PDF 처리 → 인덱스 업로드

In [ ]:
all_documents = []

for blob in pdf_blobs:
    print(f'\n{"="*60}')
    print(f'처리 중: {blob.name}')
    print(f'{"="*60}')

    pdf_bytes = pdf_container.get_blob_client(blob.name).download_blob().readall()
    documents = processor.process_pdf(pdf_bytes, blob.name)
    all_documents.extend(documents)

    # 파일 단위로 즉시 인덱스 업로드 (메모리 절약)
    if documents:
        result = mm_index.upload_documents(documents)
        text_cnt = sum(1 for d in documents if d['content_type'] == 'text')
        img_cnt  = sum(1 for d in documents if d['content_type'] == 'image')
        print(f"  → 인덱스 업로드: 텍스트 {text_cnt}개, 이미지 {img_cnt}개 "
              f"(성공={result['succeeded']}, 실패={result['failed']})")

print(f'\n전체 처리 완료: {len(all_documents)}개 문서 (PDF {len(pdf_blobs)}개)')

## 5. 멀티모달 검색 테스트

### 4.1 키워드 검색

In [ ]:
# 키워드 검색 (텍스트 + 이미지 설명 모두 포함)
query = 'single line diagram'
results = mm_index.search(query=query, top=5)

print(f'검색어: "{query}"')
print(f'검색 결과: {len(results)}건\n')

for i, doc in enumerate(results):
    icon = '🖼️' if doc['content_type'] == 'image' else '📝'
    print(f'{i+1}. {icon} [{doc["content_type"]}] {doc["document_title"]} (p.{doc["page_number"]})')
    print(f'   Score: {doc["score"]:.4f}')
    print(f'   {doc["content_text"][:150]}...')
    if doc['image_path']:
        print(f'   Image: {doc["image_path"]}')
    print()

### 4.2 하이브리드 검색 (키워드 + 벡터)

In [ ]:
from src.preprocessing.embedding import EmbeddingGenerator

emb_gen = EmbeddingGenerator(
    endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    api_key=os.environ.get('AZURE_OPENAI_API_KEY'),
    deployment=os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large'),
)

query = '전기 단선도 구성 요소'
query_embedding = emb_gen.generate(query)

results = mm_index.search(query=query, embedding=query_embedding, top=5)

print(f'하이브리드 검색어: "{query}"')
print(f'검색 결과: {len(results)}건\n')

for i, doc in enumerate(results):
    icon = '[이미지]' if doc['content_type'] == 'image' else '[텍스트]'
    print(f'{i+1}. {icon} {doc["document_title"]} (p.{doc["page_number"]})')
    print(f'   Score: {doc["score"]:.4f}')
    print(f'   {doc["content_text"][:150]}...')
    print()

### 4.3 콘텐츠 타입별 필터 검색

In [ ]:
# 이미지만 검색
query = 'diagram'
query_embedding = emb_gen.generate(query)

image_results = mm_index.search(
    query=query,
    embedding=query_embedding,
    top=5,
    content_type='image',
)

print(f'이미지 전용 검색: "{query}"')
print(f'검색 결과: {len(image_results)}건\n')

for i, doc in enumerate(image_results):
    print(f'{i+1}. 🖼️ {doc["document_title"]} (p.{doc["page_number"]})')
    print(f'   Score: {doc["score"]:.4f}')
    print(f'   {doc["content_text"][:200]}...')
    print(f'   Image: {doc["image_path"]}')
    print()

## 5. 멀티모달 RAG

검색 결과(텍스트 + 이미지 설명)를 컨텍스트로 사용하여 GPT-5.4로 응답을 생성합니다.

In [ ]:
from openai import AzureOpenAI

openai_client = AzureOpenAI(
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    api_key=os.environ.get('AZURE_OPENAI_API_KEY'),
    api_version='2024-10-21',
)

def multimodal_rag(question: str, top_k: int = 5) -> str:
    """멀티모달 RAG: 검색(텍스트+이미지) → GPT-5.4 응답"""
    # 1. 질문 임베딩
    q_embedding = emb_gen.generate(question)
    
    # 2. 하이브리드 검색 (텍스트 + 이미지 모두)
    results = mm_index.search(query=question, embedding=q_embedding, top=top_k)
    
    # 3. 컨텍스트 구성
    context_parts = []
    for i, doc in enumerate(results):
        content_type = doc['content_type']
        if content_type == 'image':
            context_parts.append(
                f'[{i+1}] (이미지, {doc["document_title"]}, p.{doc["page_number"]})\n'
                f'{doc["content_text"]}'
            )
        else:
            context_parts.append(
                f'[{i+1}] ({doc["document_title"]}, p.{doc["page_number"]})\n'
                f'{doc["content_text"]}'
            )
    
    context = '\n\n'.join(context_parts)
    
    # 4. GPT-5.4 응답 생성
    response = openai_client.chat.completions.create(
        model=os.environ.get('AZURE_OPENAI_GPT54_DEPLOYMENT', 'gpt-5.4'),
        messages=[
            {
                'role': 'system',
                'content': (
                    '주어진 컨텍스트(텍스트 및 이미지 설명)를 기반으로 질문에 답변하세요. '
                    '답변에는 출처 번호([1], [2] 등)를 포함하여 근거를 명시하세요. '
                    '이미지에서 얻은 정보도 적극 활용하세요.'
                ),
            },
            {
                'role': 'user',
                'content': f'컨텍스트:\n{context}\n\n질문: {question}',
            },
        ],
        max_tokens=1000,
        temperature=0.3,
    )
    
    return response.choices[0].message.content

print('멀티모달 RAG 함수 준비 완료')

In [ ]:
# 멀티모달 RAG 테스트
question = 'single line diagram에 포함된 주요 구성 요소와 그 역할을 설명해주세요.'

print(f'질문: {question}\n')
print('='*60)
answer = multimodal_rag(question)
print(answer)

In [ ]:
# 다이어그램/이미지 관련 질문 테스트
question2 = '문서에 포함된 다이어그램이나 도면에서 어떤 정보를 확인할 수 있나요?'

print(f'질문: {question2}\n')
print('='*60)
answer2 = multimodal_rag(question2)
print(answer2)

## 7. 결과 확인

In [ ]:
# 인덱스 통계
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential

idx_client = SearchIndexClient(
    endpoint=os.environ['AZURE_SEARCH_SERVICE_ENDPOINT'],
    credential=AzureKeyCredential(os.environ['AZURE_SEARCH_ADMIN_KEY']) if os.environ.get('AZURE_SEARCH_ADMIN_KEY') else DefaultAzureCredential(),
)

stats = idx_client.get_index_statistics(MULTIMODAL_INDEX_NAME)
print(f'인덱스: {MULTIMODAL_INDEX_NAME}')
print(f'  문서 수: {stats.document_count}')
print(f'  저장 크기: {stats.storage_size:,} bytes')

In [ ]:
# Blob Storage에 저장된 추출 이미지 확인
processed_container = blob_service.get_container_client('processed-documents')
image_blobs = [b for b in processed_container.list_blobs(name_starts_with='multimodal/') if b.name.endswith('.png')]

print(f'추출된 이미지 파일 {len(image_blobs)}개:')
for blob in image_blobs:
    print(f'  - {blob.name} ({blob.size:,} bytes)')

## 정리

이 노트북에서 구현한 내용:

| 단계 | 설명 | Azure 서비스 |
|------|------|-------------|
| 문서 분석 | PDF에서 텍스트 + 이미지 추출 | Document Intelligence (Layout) |
| Image Verbalization | 이미지 → 자연어 설명 | GPT-5.4 |
| 임베딩 | 텍스트/이미지 설명 벡터화 | text-embedding-3-large |
| 이미지 저장 | 추출 이미지 Blob 저장 | Blob Storage |
| 인덱싱 | 멀티모달 벡터 인덱스 | AI Search |
| 검색 | 하이브리드 + 필터 검색 | AI Search |
| RAG | 검색 결과 기반 응답 생성 | GPT-5.4 |

### 두 가지 멀티모달 임베딩 방식 비교

| 방식 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **Image Verbalization** (본 랩) | GPT-5.4로 이미지→텍스트 변환 후 텍스트 임베딩 | 다이어그램/차트의 의미적 관계 파악, RAG 인용 가능 | LLM 호출 비용, 인덱싱 시간 증가 |
| **Direct Multimodal Embedding** | Cohere 등 멀티모달 모델로 이미지/텍스트 직접 임베딩 | 설정 간편, LLM 불필요 | 이미지 유사도만 반영, 의미적 설명 불가 |